# 06  -  Domain Adaptation (zero-shot, few-shot, full)

Three clearly separated transfer regimes, run for all 6 source -> target pairs. The methods are **Source-only** (no adaptation),
**DANN**, **Subdomain Adaptation** (MMD) and **HEN** (attention network), compared with a **Target-only RF** trained from scratch on the same labels.

| Regime | Target labels used for training | Question |
|---|---|---|
| **zero-shot** | none (unlabelled target features may be used by DANN / MMD) | does anything transfer at all? |
| **few-shot** | `FEW_SHOT_LABELS` fraud cases (+10x legitimate) from the target *train* rows | does transfer beat training from scratch on the same few labels? |
| **full** | all target train rows (Target-only RF; reference ceiling) | how far is transfer from just having labels? |

Protocol rules: model selection uses **source validation** only; the target test rows (NB02 split) are touched once, at the end;
the operating point is "flag the top `REVIEW_BUDGET` of records" so no target labels are needed to set a threshold;
the headline metrics are PR-AUC and recall at that budget (F1/MCC at the same operating point are kept for compatibility).

**Inputs:** `outputs/*_features.csv` (with the `split` column)  
**Outputs:** `outputs/domain_adaptation_results.csv`

In [ ]:
exec(open('00_config.py').read())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef,
                              precision_recall_curve, auc as sklearn_auc)
from imblearn.over_sampling import SMOTE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DA_EPOCHS   = 20          # source epochs per model (DANN_EPOCHS in the config is for the full-size run)
MAX_SOURCE  = 100_000     # source rows per training run (all frauds kept)
MAX_TARGET_UNLABELLED = 30_000
FEW_SHOT_SEEDS = [0, 1, 2]
print(f'Device: {DEVICE}')

frames = {'Credit Card': pd.read_csv(CC_FEATURES_CSV), 'Mobile Money': pd.read_csv(MM_FEATURES_CSV),
          'Health Insurance': pd.read_csv(HI_FEATURES_CSV)}
for d, f in frames.items():
    print(f'{d:<18} {f.shape}  split counts: {f[SPLIT_COL].value_counts().to_dict()}')

## Helpers

In [ ]:
def subsample(X, y, max_n=MAX_SOURCE, seed=RANDOM_STATE):
    """Keep every fraud row when possible; thin the legitimate rows."""
    if len(X) <= max_n: return X, y
    rng = np.random.default_rng(seed)
    fi, ni = np.where(y == 1)[0], np.where(y == 0)[0]
    nf = min(len(fi), max_n // 5)
    idx = np.concatenate([rng.choice(fi, nf, replace=False), rng.choice(ni, max_n - nf, replace=False)])
    rng.shuffle(idx)
    return X.iloc[idx], y.iloc[idx]

def pr_auc_score(y, p):
    prec, rec, _ = precision_recall_curve(y, p)
    return sklearn_auc(rec, prec)

def evaluate_at_budget(y_true, probs, src, tgt, method, regime, n_labels, seed=0):
    """Headline metrics plus the fixed-budget operating point (flag the top REVIEW_BUDGET of target test records)."""
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    k = max(1, int(round(REVIEW_BUDGET * len(y_true))))
    flag = np.zeros(len(y_true), dtype=int); flag[np.argsort(-probs, kind='stable')[:k]] = 1
    m = {'regime': regime, 'n_labels': n_labels, 'seed': seed, 'method': method, 'source': src, 'target': tgt,
         'f1': f1_score(y_true, flag, zero_division=0), 'precision': precision_score(y_true, flag, zero_division=0),
         'recall': recall_score(y_true, flag, zero_division=0), 'mcc': matthews_corrcoef(y_true, flag),
         'recall_at_budget': recall_at_budget(y_true, probs), 'base_rate': y_true.mean()}
    try:
        m['roc_auc'] = roc_auc_score(y_true, probs); m['pr_auc'] = pr_auc_score(y_true, probs)
    except Exception:
        m['roc_auc'] = m['pr_auc'] = np.nan
    return m

def maybe_smote(X, y, target=0.10):
    rate = y.mean()
    if rate >= target: return X, y
    desired = 0.05 if rate < 0.01 else 0.10 if rate < 0.05 else target
    k = min(5, int(y.sum()) - 1)
    if k < 1: return X, y
    return SMOTE(sampling_strategy=desired, random_state=RANDOM_STATE, k_neighbors=k).fit_resample(X, y)

def pos_weight(y):
    n_pos = (y == 1).sum()
    return torch.tensor((y == 0).sum() / n_pos if n_pos > 0 else 1.0, dtype=torch.float32)

print('Helpers defined')

## Shared feature space

Only two things are shared across domains: (1) three **concept scores** (amount anomaly, counterparty risk, structural distance) built from keyword-matched features, and (2) features that have **the same name** in both domains. Earlier versions aligned unrelated columns by position, which is not meaningful. Every standardisation uses the domain's own *train* rows.

In [ ]:
SEMANTIC_GROUPS = {
    'Credit Card': {
        'amount_anomaly':      ['amount_zscore','amount_deviation','is_large','is_micro','amt_ghs','is_high_value','amount_is_round','amt'],
        'counterparty_risk':   ['category','merchant','high_risk_cat','high_risk_merch'],
        'structural_distance': ['geo_distance','long_distance','is_rural','unusual_timing','tx_burst','rapid_succession'],
    },
    'Mobile Money': {
        'amount_anomaly':      ['amount_zscore','amount_deviation','is_large','is_micro','is_high_value','amount'],
        'counterparty_risk':   ['recipient','oldBal','newBal','balance_zeroed','is_agent'],
        'structural_distance': ['balance_error','balance_diff','unusual_timing','tx_burst','rapid_succession'],
    },
    'Health Insurance': {
        'amount_anomaly':      ['amount_zscore','amount_deviation','is_large','is_micro','is_high_value','total_ghc','services_ghc','pharmacy_ghc','dev_total','dev_service','dev_medical','max_deviation'],
        'counterparty_risk':   ['gdrg','icd10','drg_fraud_rate','Diagnosis','G-DRG','drg_claim','drg_billing','drg_cluster'],
        'structural_distance': ['length_of_stay','overbill','multi_field','billing_variab','unusual_timing','tx_burst','rapid_succession'],
    },
}
CONCEPTS = ['amount_anomaly','counterparty_risk','structural_distance']

def concept_scores(X, ref, domain):
    """X, ref: DataFrames with the same columns. z-scores use `ref` (train rows) statistics."""
    out = np.zeros((len(X), 3), dtype=np.float32)
    for j, concept in enumerate(CONCEPTS):
        kws = SEMANTIC_GROUPS[domain][concept]
        cols = [c for c in X.columns if any(k.lower() in c.lower() for k in kws)]
        if cols:
            mu, sd = ref[cols].mean(), ref[cols].std() + 1e-8
            out[:, j] = ((X[cols] - mu) / sd).mean(axis=1).values
    return out

def shared_names(src, tgt):
    s, t = set(frames[src].columns) - {TARGET_COL, SPLIT_COL}, set(frames[tgt].columns) - {TARGET_COL, SPLIT_COL}
    return sorted(s & t)

def build_space(domain, other, parts):
    """parts = dict name -> DataFrame (features only). Returns shared-space arrays scaled with the domain's TRAIN statistics."""
    names = shared_names(domain, other)
    ref = parts['train']
    sc = StandardScaler().fit(np.hstack([concept_scores(ref, ref, domain), ref[names].values]) if names else concept_scores(ref, ref, domain))
    out = {}
    for k, X in parts.items():
        Z = np.hstack([concept_scores(X, ref, domain), X[names].values]) if names else concept_scores(X, ref, domain)
        out[k] = sc.transform(Z).astype(np.float32)
    return out

def pair_data(src, tgt):
    sXtr, sXva, _, sytr, syva, _ = split_frames(frames[src])
    tXtr, _, tXte, tytr, _, tyte = split_frames(frames[tgt])
    S = build_space(src, tgt, {'train': sXtr, 'val': sXva})
    T = build_space(tgt, src, {'train': tXtr, 'test': tXte})
    sXtr_s, sytr_s = subsample(pd.DataFrame(S['train']), sytr)
    return dict(sX=sXtr_s.values, sy=sytr_s.values.astype(np.float32), sXva=S['val'], syva=syva.values,
                tXtr=T['train'], tytr=tytr.values, tXte=T['test'], tyte=tyte.values, dim=S['train'].shape[1])

print('Shared-space construction defined')

## Models

One MLP backbone; the adaptation method only changes the extra loss on **unlabelled** target features:
Source-only (none), DANN (gradient-reversal domain classifier), Subdomain Adaptation (MMD between source and target features), HEN (attention gate before the encoder, no extra loss).

In [ ]:
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.clone()
    @staticmethod
    def backward(ctx, grad):  return -ctx.lam * grad, None

class Net(nn.Module):
    def __init__(self, d, method):
        super().__init__()
        self.method = method; self.lam = 0.0
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Tanh(), nn.Linear(d, d), nn.Softmax(dim=-1)) if method == 'HEN' else None
        h1, h2 = DANN_HIDDEN_DIMS
        self.fe  = nn.Sequential(nn.Linear(d, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(0.3))
        self.clf = nn.Sequential(nn.Linear(h2, 64), nn.ReLU(), nn.Linear(64, 1))
        self.dom = nn.Sequential(nn.Linear(h2, 64), nn.ReLU(), nn.Linear(64, 2)) if method == 'DANN' else None
    def features(self, x):
        if self.gate is not None: x = x * self.gate(x) * x.shape[1]      # rescale so the gate starts near identity
        return self.fe(x)
    def forward(self, x):
        f = self.features(x)
        return self.clf(f).squeeze(-1), f

def mmd(s, t, bw=1.0):
    def k(X, Y):
        d = (X**2).sum(1, keepdim=True) + (Y**2).sum(1, keepdim=True).T - 2*X@Y.T
        return torch.exp(-d / (2*bw**2))
    return k(s, s).mean() + k(t, t).mean() - 2*k(s, t).mean()

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.FloatTensor(X).to(DEVICE))[0]).cpu().numpy()

def train_source(method, D, seed=RANDOM_STATE):
    """Train on source train rows; select the epoch by SOURCE validation PR-AUC. Target labels are never used."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    sX, sy = maybe_smote(pd.DataFrame(D['sX']), pd.Series(D['sy'])); sX, sy = sX.values.astype(np.float32), sy.values.astype(np.float32)
    tu = D['tXtr'][rng.choice(len(D['tXtr']), min(MAX_TARGET_UNLABELLED, len(D['tXtr'])), replace=False)]   # features only
    model = Net(D['dim'], method).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=DANN_LR, weight_decay=1e-4)
    clf_fn, dom_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight(sy).to(DEVICE)), nn.CrossEntropyLoss()
    bs = 256
    s_loader = DataLoader(TensorDataset(torch.FloatTensor(sX), torch.FloatTensor(sy)), batch_size=bs, shuffle=True, drop_last=True)
    best, best_state = -1, None
    for ep in range(DA_EPOCHS):
        model.train(); p = ep / DA_EPOCHS
        lam = (2 / (1 + np.exp(-10 * p)) - 1) * DANN_LAMBDA_MAX
        for sx, sy_b in s_loader:
            sx, sy_b = sx.to(DEVICE), sy_b.to(DEVICE)
            tx = torch.FloatTensor(tu[rng.choice(len(tu), bs, replace=False)]).to(DEVICE)
            logits, fs = model(sx); loss = clf_fn(logits, sy_b)
            if method == 'DANN':
                ft = model.features(tx)
                ds, dt = model.dom(GRL.apply(fs, lam)), model.dom(GRL.apply(ft, lam))
                loss = loss + lam * (dom_fn(ds, torch.zeros(len(sx), dtype=torch.long, device=DEVICE)) +
                                     dom_fn(dt, torch.ones(len(tx), dtype=torch.long, device=DEVICE))) / 2
            elif method == 'Subdomain Adaptation':
                loss = loss + 0.1 * mmd(fs, model.features(tx))
            opt.zero_grad(); loss.backward(); opt.step()
        score = pr_auc_score(D['syva'], predict(model, D['sXva'])) if D['syva'].sum() > 0 else 0
        if score > best: best, best_state = score, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state or model.state_dict())
    return model

def few_shot_indices(y, n_pos, seed, legit_ratio=10):
    """n_pos fraud rows + legit_ratio*n_pos legitimate rows from the target TRAIN split."""
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n_pos = min(n_pos, len(pos))
    return np.concatenate([rng.choice(pos, n_pos, replace=False), rng.choice(neg, min(len(neg), legit_ratio * n_pos), replace=False)])

def fine_tune(model, X, y, epochs=30, lr=3e-4):
    import copy
    m = copy.deepcopy(model); m.train()
    opt = optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight(y).to(DEVICE))
    loader = DataLoader(TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y)), batch_size=min(64, len(X)), shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            if len(xb) < 2: continue
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss_fn(m(xb)[0], yb).backward(); opt.step()
    return m

def target_only_rf(D, idx=None, seed=RANDOM_STATE):
    X, y = (D['tXtr'], D['tytr']) if idx is None else (D['tXtr'][idx], D['tytr'][idx])
    if idx is None and len(X) > 300_000:                      # full-label reference, thinned for speed
        keep = np.random.default_rng(seed).choice(len(X), 300_000, replace=False); X, y = X[keep], y[keep]
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', min_samples_leaf=3, random_state=seed, n_jobs=-1)
    return rf.fit(X, y).predict_proba(D['tXte'])[:, 1]

print('Models defined')

## Run all transfer scenarios (zero-shot, few-shot, full)

In [ ]:
scenarios = [('Credit Card','Mobile Money'), ('Credit Card','Health Insurance'),
             ('Mobile Money','Credit Card'), ('Mobile Money','Health Insurance'),
             ('Health Insurance','Credit Card'), ('Health Insurance','Mobile Money')]
METHODS = ['Source-only', 'DANN', 'Subdomain Adaptation', 'HEN']

da_results = []
os.makedirs(MODEL_DIR, exist_ok=True)
for src, tgt in scenarios:
    print(f'\n===== {src} -> {tgt} =====')
    D = pair_data(src, tgt)
    print(f'  shared dim={D["dim"]}  source train={len(D["sX"]):,}  target test={len(D["tyte"]):,} ({int(D["tyte"].sum())} frauds)')
    for method in METHODS:
        model = train_source(method, D)
        key = f'{src[:2].upper()}_to_{tgt[:2].upper()}_{method.split()[0].lower()}'
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'da_{key}.pt'))
        # --- zero-shot: no target labels
        r = evaluate_at_budget(D['tyte'], predict(model, D['tXte']), src, tgt, method, 'zero-shot', 0)
        da_results.append(r); print(f'  [zero-shot] {method:<22} PR-AUC={r["pr_auc"]:.4f}  recall@{REVIEW_BUDGET:.0%}={r["recall_at_budget"]:.3f}  (base rate {r["base_rate"]:.3%})')
        # --- few-shot: fine-tune on n labelled target fraud cases (+10x legit), several random draws
        for n in FEW_SHOT_LABELS:
            for seed in FEW_SHOT_SEEDS:
                idx = few_shot_indices(D['tytr'], n, seed)
                ft = fine_tune(model, D['tXtr'][idx], D['tytr'][idx].astype(np.float32))
                da_results.append(evaluate_at_budget(D['tyte'], predict(ft, D['tXte']), src, tgt, method, 'few-shot', n, seed))
    # --- from-scratch references on the same target labels
    for n in FEW_SHOT_LABELS:
        for seed in FEW_SHOT_SEEDS:
            idx = few_shot_indices(D['tytr'], n, seed)
            if D['tytr'][idx].sum() >= 1:
                da_results.append(evaluate_at_budget(D['tyte'], target_only_rf(D, idx, seed), src, tgt, 'Target-only RF', 'few-shot', n, seed))
    da_results.append(evaluate_at_budget(D['tyte'], target_only_rf(D), src, tgt, 'Target-only RF', 'full', int(D['tytr'].sum())))

da_df = pd.DataFrame(da_results)
da_df.to_csv(os.path.join(OUTPUT_DIR, 'domain_adaptation_results.csv'), index=False)
print('\nSaved results ->', os.path.join(OUTPUT_DIR, 'domain_adaptation_results.csv'))

## Summary (mean over scenarios and draws)

In [ ]:
cols = ['pr_auc','recall_at_budget','roc_auc']
summary = (da_df.groupby(['regime','n_labels','method'])[cols].mean().round(4))
print(summary.to_string())
print(f'\nChance level: recall@{REVIEW_BUDGET:.0%} = {REVIEW_BUDGET:.2f}; PR-AUC = the target fraud rate (see base_rate column).')

# Lift of PR-AUC over each target's base rate, so domains with different fraud rates are comparable
da_df['pr_lift'] = da_df['pr_auc'] / da_df['base_rate']
print('\nMean PR-AUC lift over base rate:')
print(da_df.groupby(['regime','n_labels','method'])['pr_lift'].mean().round(2).to_string())

In [ ]:
domain_abbrev = {'Credit Card':'CC','Mobile Money':'MM','Health Insurance':'HI'}
da_df['src'] = da_df['source'].map(domain_abbrev); da_df['tgt'] = da_df['target'].map(domain_abbrev)

# 1) zero-shot heatmaps (PR-AUC lift) per method
zs = da_df[da_df['regime'] == 'zero-shot']
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
for ax, method in zip(axes, METHODS):
    pv = zs[zs['method'] == method].pivot(index='src', columns='tgt', values='pr_lift')
    sns.heatmap(pv, annot=True, fmt='.1f', cmap='RdYlGn', center=1, ax=ax, annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(f'{method}\nzero-shot PR-AUC lift (1 = chance)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'domain_adaptation_heatmaps.png'), dpi=150, bbox_inches='tight')
plt.show()

# 2) label-efficiency curves: does transfer beat training from scratch on the same labels?
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=False)
for ax, tgt in zip(axes, ['Credit Card', 'Mobile Money', 'Health Insurance']):
    sub = da_df[(da_df['target'] == tgt)]
    for method in METHODS + ['Target-only RF']:
        s = sub[sub['method'] == method]
        zero = s[s['regime'] == 'zero-shot']['pr_lift'].mean() if method != 'Target-only RF' else np.nan
        few = s[s['regime'] == 'few-shot'].groupby('n_labels')['pr_lift'].mean()
        xs = ([0] if not np.isnan(zero) else []) + list(few.index)
        ys = ([zero] if not np.isnan(zero) else []) + list(few.values)
        ax.plot(xs, ys, marker='o', label=method, linestyle='--' if method == 'Target-only RF' else '-')
    full = sub[(sub['method'] == 'Target-only RF') & (sub['regime'] == 'full')]['pr_lift'].mean()
    ax.axhline(full, color='k', linestyle=':', label='Target-only RF, all labels')
    ax.set_xscale('symlog', linthresh=10); ax.set_xlabel('labelled target fraud cases'); ax.set_ylabel('PR-AUC lift over base rate')
    ax.set_title(f'target = {tgt}', fontweight='bold')
axes[0].legend(fontsize=8)
plt.suptitle('Label efficiency: zero-shot (x=0), few-shot, full', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'domain_adaptation_label_efficiency.png'), dpi=150, bbox_inches='tight')
plt.show()